# Scanned Chapter to Multilingual Narrated Lesson (`bodhan_lesson`)

This notebook demonstrates the end-to-end pipeline that transforms scanned textbook chapters into pedagogical, multilingual narrated lessons using **Bodhan Indic AI** models:

1. **Indic-OCR**: Reading-ordered text, table, and LaTeX extraction.
2. **Indic-Translate**: Structure-preserving cross-lingual translation with LaTeX shielding.
3. **Indic-Speak**: Natural educational speech synthesis normalized to **-16 LUFS**.
4. **Indic-Transcribe**: 3-dimensional audio quality verification (Coverage, Repetition, CER/WER).

## 1. Setup and Environment Configuration

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# Set mock mode if testing offline
# os.environ["BODHAN_MOCK"] = "1"

print("Base URL:", os.environ.get("BODHAN_BASE_URL", "https://api.bodhan.ai"))

## 2. Ingestion & Document Intelligence (Layer 1)

Rasterize input scanned pages at native 300 DPI, extract OCR blocks, filter running noise, and build the typed **Document IR**.

In [ ]:
from bodhan_lesson.assembler import assembler
from bodhan_lesson.ingestion import ingest_scans_to_page_images
from bodhan_lesson.ocr import ocr_processor

input_path = Path("sample_data/ncert_lines_and_angles")
source_dir = Path("out/source")
source_dir.mkdir(parents=True, exist_ok=True)

# 1. Ingestion
page_images = ingest_scans_to_page_images(input_path, source_dir)
print(f"Ingested {len(page_images)} page(s).")

# 2. OCR
page_results = ocr_processor.process_document_pages(page_images)

# 3. Assembly into Document IR
doc_ir = assembler.assemble_document(page_results, source_language="en")
print(f"Document IR created: {len(doc_ir.sections)} sections assembled.")
print(f"Source lesson preview:\n{doc_ir.source_markdown[:300]}...")

## 3. Language Intelligence (Layer 2)

Shield LaTeX formulas and tables, translate lesson content into target Indic languages, and generate pedagogical narration plans.

In [ ]:
from bodhan_lesson.narration import narration_planner
from bodhan_lesson.translator import translator

target_lang = "hi"

# Translate full lesson markdown with token shielding
translated_md = translator.translate_markdown(
    text=doc_ir.source_markdown,
    target_lang=target_lang,
    source_lang="en",
)
print(f"Translated Markdown preview ({target_lang}):\n{translated_md[:300]}...")

# Generate grounded narration for first section
sec = doc_ir.sections[0]
narration = narration_planner.generate_narration(sec, target_lang=target_lang, source_lang="en")
print(f"English Spoken Script:\n{narration.english_script}")
print(f"Target Spoken Script ({target_lang}):\n{narration.target_script}")

## 4. Audio Intelligence (Layer 3)

Synthesize speech with sentence chunking, normalize integrated loudness to **-16 LUFS**, and verify with **Indic-Transcribe**.

In [ ]:
from bodhan_lesson.synthesizer import synthesizer
from bodhan_lesson.verifier import audio_verifier

out_wav = Path("out/hi/01-preview.hi.wav")
out_mp3 = Path("out/hi/01-preview.hi.mp3")

duration, lufs = synthesizer.synthesize_section(
    script_text=narration.target_script,
    lang=target_lang,
    output_wav_path=out_wav,
    output_mp3_path=out_mp3,
    target_lufs=-16.0,
)
print(f"Synthesized: {duration:.2f}s at {lufs:.2f} LUFS")

# 3D Audio QA
qa = audio_verifier.verify_section_audio(
    audio_bytes=out_wav.read_bytes(),
    reference_script=narration.target_script,
    section_id="01",
    language=target_lang,
)
print(f"QA Results: Coverage={qa.coverage_score:.2f}, CER={qa.cer:.2f}, WER={qa.wer:.2f}, Passed={qa.passed}")

## 5. End-to-End CLI Invocation

To run the complete pipeline over any scanned chapter from your terminal:

```bash
python -m bodhan_lesson --input sample_data/ncert_lines_and_angles --langs hi,ta,bn --out ./out
```